# E-Commerce Sales Prediction
**Author:** Angel Vazquez Maldonado  
**GitHub:** github.com/Avazquez0913

## Project Overview
This project builds and compares multiple machine learning models to predict **Units Sold** in an e-commerce dataset.

The pipeline covers:
1. Data loading and exploration (EDA)
2. Feature engineering
3. Data preprocessing with proper train/test split
4. Model training and comparison (5 models)
5. Feature importance analysis
6. Results visualization

**Models compared:** Linear Regression, Decision Tree, Random Forest, K-Nearest Neighbors, Neural Network (MLP)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully')

## 2. Load Dataset

In [ ]:
# Load dataset — update path as needed
df = pd.read_csv('Ecommerce_Sales_Prediction_Dataset.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nData types:')
print(df.dtypes)

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Distribution of target variable: Units Sold
plt.figure(figsize=(8, 5))
sns.histplot(df['Units_Sold'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Units Sold', fontsize=14)
plt.xlabel('Units Sold')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Units Sold vs Marketing Spend
plt.figure(figsize=(8, 5))
plt.scatter(df['Marketing_Spend'], df['Units_Sold'], alpha=0.4, color='steelblue')
plt.title('Units Sold vs Marketing Spend', fontsize=14)
plt.xlabel('Marketing Spend')
plt.ylabel('Units Sold')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
# Convert Date to datetime and extract Month
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Month'] = df['Date'].dt.month

# Engineer Revenue feature
df['Revenue'] = df['Price'] * df['Units_Sold']

print('New features added: Month, Revenue')
df[['Date', 'Month', 'Revenue']].head()

## 5. Preprocessing — Encoding & Scaling

> **Note:** Encoding and scaling are fit **only on training data** to prevent data leakage.

In [ ]:
# Encode categorical variables
le = LabelEncoder()
df['Product_Category_Enc'] = le.fit_transform(df['Product_Category'])
df['Customer_Segment_Enc'] = le.fit_transform(df['Customer_Segment'])

# Define features and target
features = ['Product_Category_Enc', 'Price', 'Discount',
            'Customer_Segment_Enc', 'Marketing_Spend', 'Month', 'Revenue']
target = 'Units_Sold'

X = df[features]
y = df[target]

# Train/test split — 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing samples:  {X_test.shape[0]}')

In [ ]:
# Scale features — fit ONLY on training data to prevent leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # transform only — no fit

print('Scaling complete — scaler fit on training data only')

## 6. Correlation Analysis

In [ ]:
# Correlation heatmap — features only (exclude target to avoid bias)
plt.figure(figsize=(10, 6))
corr_matrix = df[features].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Model Training & Comparison

In [ ]:
# Define models
models = {
    'Linear Regression':   LinearRegression(),
    'Decision Tree':       DecisionTreeRegressor(random_state=42),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42),
    'K-Nearest Neighbors': KNeighborsRegressor(n_neighbors=5),
    'Neural Network (MLP)': MLPRegressor(hidden_layer_sizes=(64, 32),
                                          max_iter=1000, random_state=42)
}

# Train each model and evaluate
results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    mae  = mean_absolute_error(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_test, y_pred)

    results[name] = {'MAE': round(mae, 4), 'MSE': round(mse, 4),
                     'RMSE': round(rmse, 4), 'R2': round(r2, 4)}
    print(f'{name}: MAE={mae:.4f} | RMSE={rmse:.4f} | R2={r2:.4f}')

results_df = pd.DataFrame(results).T.sort_values('MAE')
print('\n--- Model Comparison (sorted by MAE) ---')
print(results_df)

## 8. Results Visualization

In [ ]:
# Bar chart comparing MAE across models
plt.figure(figsize=(10, 5))
plt.bar(results_df.index, results_df['MAE'], color='steelblue', edgecolor='black')
plt.title('Model Comparison — Mean Absolute Error (lower is better)', fontsize=13)
plt.ylabel('MAE')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 9. Feature Importance (Random Forest)

In [ ]:
# Extract feature importance from Random Forest
rf_model = models['Random Forest']
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x='Importance', y='Feature', palette='Blues_r')
plt.title('Feature Importance — Random Forest', fontsize=13)
plt.tight_layout()
plt.show()

print(importance_df.to_string(index=False))

## 10. Summary

This project demonstrated a complete ML pipeline:

- **EDA** — explored distributions, correlations, and relationships between features
- **Feature Engineering** — created Revenue and Month features to improve model signal
- **Proper preprocessing** — encoder and scaler fit only on training data to prevent data leakage
- **Model comparison** — evaluated 5 algorithms using MAE, RMSE, and R2
- **Feature importance** — identified which variables most influence sales volume

**Key finding:** Revenue (Price × Units_Sold) and Marketing_Spend were the most influential features — suggesting that pricing strategy and marketing investment are the primary drivers of sales volume in this dataset.

---
**Author:** Angel Vazquez Maldonado | github.com/Avazquez0913